# CrewAI API Tutorial

This notebook introduces the core CrewAI abstractions used to create a
multi-agent workflow:

- `Agent`: defines a specialized AI team member.
- `Task`: describes work assigned to an agent.
- `Crew`: groups agents and tasks into one workflow.
- `Process`: determines the order in which tasks execute.

We will build a sequential two-agent crew. A technical researcher will prepare
notes about a topic, and a technical writer will use those notes to produce a
beginner-friendly explanation.

## 1. Import the reusable utilities

The project keeps common CrewAI setup logic in `crewai_utils.py`. This avoids
repeating agent, task, and crew construction code across notebooks.

In [ ]:
import os

import crewai_utils as crut

## 2. Create specialized agents

Each agent has a role, goal, and backstory. These fields guide how the language
model approaches its assigned work.

The model name is read from the `CREWAI_MODEL` environment variable. API keys
must also be supplied through environment variables and should never be written
directly in the notebook.

In [ ]:
llm = os.getenv("CREWAI_MODEL")

researcher = crut.create_agent(
    role="Technical Researcher",
    goal="Prepare accurate and concise technical notes about a given topic.",
    backstory=(
        "You analyze technical subjects and organize the important ideas "
        "before they are presented to a general audience."
    ),
    llm=llm,
    verbose=True,
)

writer = crut.create_agent(
    role="Technical Writer",
    goal="Turn technical research into a clear beginner-friendly explanation.",
    backstory=(
        "You specialize in explaining complex technical ideas using simple "
        "language and practical examples."
    ),
    llm=llm,
    verbose=True,
)

## 3. Define the tasks

The research task runs first. The writing task lists the research task as
context, allowing the writer to use the researcher’s output.

In [ ]:
research_task = crut.create_task(
    description=(
        "Prepare concise notes about {topic}. Include its purpose, core "
        "concepts, and one practical use case."
    ),
    expected_output=(
        "A structured list of technical notes containing the purpose, core "
        "concepts, and one practical use case."
    ),
    agent=researcher,
)

writing_task = crut.create_task(
    description=(
        "Use the research notes to write a beginner-friendly explanation of "
        "{topic}. Keep the response under 400 words."
    ),
    expected_output=(
        "A short Markdown explanation with a title, overview, key concepts, "
        "and practical example."
    ),
    agent=writer,
    context=[research_task],
)

## 4. Assemble the sequential crew

A sequential process executes tasks in their listed order. The researcher
completes the first task, and the writer then receives that result as context.

In [ ]:
crew = crut.create_sequential_crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    verbose=True,
)

crew

## 5. Run the workflow

Choose a topic and execute the crew. Running this cell requires credentials for
the model provider configured through `CREWAI_MODEL`. For example, an OpenAI
model requires `OPENAI_API_KEY` to be available in the environment.

Do not save API keys inside this notebook or commit them to Git.

In [2]:
RUN_LLM = False
topic = "multi-agent systems"

if RUN_LLM:
    result = crut.run_crew(
        crew,
        inputs={"topic": topic},
    )
    print(result)
else:
    print(
        "Crew created successfully. Set RUN_LLM=True after configuring "
        "your model provider credentials."
    )

Crew created successfully. Set RUN_LLM=True after configuring your model provider credentials.
